<a href="https://colab.research.google.com/github/claramanolache/ML_Intro/blob/main/Week_4_Practice.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Week 4: Practice Notebook

**You do not need to submit this Notebook.** It is for practice only.

## Introduction to the Dataset

The Iris dataset is one of the most famous and widely used datasets in machine learning. It contains measurements of 150 iris flowers, divided equally among three species: Iris setosa, Iris versicolor, and Iris virginica. For each flower, four features are recorded: [sepal](https://en.wikipedia.org/wiki/Sepal) length, sepal width, [petal](https://en.wikipedia.org/wiki/Petal) length, and petal width. These measurements make the dataset ideal for learning and practicing classification algorithms, as the species can be predicted based on the flower’s physical characteristics.

### Goal
We will use linear regression to estimate one feature of iris flowers given another feature, and logistic regression to identify iris species given all of their features.

## Import Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn import datasets
from sklearn import dummy
from sklearn import linear_model
from sklearn import metrics
from sklearn import preprocessing

Let's load the Iris dataset. Since the dataset is very small, we won't split it into training and test sets for this exercise. Instead, we'll use the whole dataset to explore linear models and their behavior.

In [ ]:
iris = datasets.load_iris(as_frame=True)
X, y = iris.data, iris.target
# Modify feature names to reference them more easily.
X.columns = (name.replace(" ", "_").replace("_(cm)", "") for name in X.columns)
X.shape, y.shape

After loading the data, it's good practice to check the shape of your feature matrix and target vector. This helps confirm that the data loaded correctly and gives you a sense of how many samples and features you'll be working with.

Let's look at the first few rows of the Iris dataset to get familiar with the features and see how many samples belong to each species.

In [ ]:
X.head()

We can also see that we have exactly 50 labels for each of the three Iris species.

In [ ]:
y.value_counts()

Next, we'll create a scatter plot of the sepal and petal lengths versus widths. This helps us see if there's a relationship between these two features, and whether they might help us distinguish between different species.

In [ ]:
_, axes = plt.subplots(nrows=1, ncols=2, sharey=True)
X.plot.scatter("sepal_length", "sepal_width", c=y, colormap="viridis", colorbar=False, ax=axes[0])
X.plot.scatter("petal_length", "petal_width", c=y, colormap="viridis", colorbar=False, ax=axes[1])
axes[0].set_title("Sepal Size Comparison")
axes[1].set_title("Petal Size Comparison")

#### Self-Check
Which part of these irises - petal or sepal size - appears to best separate one species from the others?

Let's focus our attention on petal sizes for now, as it appears there is a near-linear relationship between petal length and width across species. We can attempt make petal width predictions based on length (or vice versa).

In [ ]:
X.plot.scatter("petal_length", "petal_width")

#### Self-Check
What does a linear relationship look like in a scatter plot?

We'll create a feature matrix that includes just the petal length, and a target vector that includes just the petal width.

Note how we index differently here; indexing once on a DataFrame gives us a Series (a vector), but indexing with a nested list gives us another DataFrame (a matrix).

In [ ]:
X_petal, y_petal = X[["petal_length"]], X["petal_width"]
X_petal.shape, y_petal.shape

Let's establish a baseline. Similar to the DummyClassifier we saw previously, DummyRegressor predicts the mean value of the target variable for every input. Comparing our linear regression model to this baseline helps us see if our model is actually learning something useful.

In [ ]:
dummy_reg = dummy.DummyRegressor()
dummy_reg.fit(X_petal, y_petal)
y_pred_dummy = dummy_reg.predict(X_petal)

#### Self-Check
What does a DummyRegressor do, and why is it important to compare our model to this baseline?

Recall that the Root Mean Square Error (RMSE) is a metric for evaluating regression models. It tells us, on average, how far off our predictions are from the actual values. By comparing the RMSE of this baseline to the linear regression model below, we can judge whether our model is making meaningful predictions.

In [ ]:
rmse = metrics.root_mean_squared_error(y_petal, y_pred_dummy)
rmse

Now we'll train a linear regression model to predict petal width from petal length. Linear regression finds the best-fit line through the data, minimizing the average squared error between predictions and actual values.

In [ ]:
lin_reg = linear_model.LinearRegression()
lin_reg.fit(X_petal, y_petal)
y_pred_lin = lin_reg.predict(X_petal)

In [ ]:
rmse = metrics.root_mean_squared_error(y_petal, y_pred_lin)
rmse

The linear regression model's RMSE is lower than the baseline - but is it any good? Let's look at the ranges for the petal widths and lengths.

In [ ]:
y_petal.min(), y_petal.max()

The petal widths range from 0.1 to 2.5 centimeters, so an RMSE of 0.2 is quite good, especially when compared to the baseline.

Plotting both the baseline and the linear regression predictions helps us visually compare their performance. The baseline will appear as a horizontal line (the mean), while the linear regression line should follow the trend in the data.

In [ ]:
X.plot.scatter("petal_length", "petal_width")
plt.plot(X_petal, y_pred_lin, c="red")
plt.plot(X_petal, y_pred_dummy, c="orange")

#### Self-Check
How does the linear regression line differ from the baseline in the plot, and what does the slope represent?

## Logistic Regression

Linear regression is great for predicting numbers, but what if we want to predict categories, like which species a flower belongs to? For that, we use logistic regression, which is designed for classification problems.

#### Self-Check
What is the main difference between regression and classification problems?

### Binary Logistic Regression (Setosa vs. Not Setosa)

To start, we'll simplify the problem to a binary classification: predicting whether a flower is Setosa or not. This makes it easier to understand how logistic regression works before moving on to multiclass classification.

In the plot at the beginning of this notebook, we could see that one species of flower had more distance from the other two iris species; this species is Iris setosa.

Let's create a binary label vector that will identify Iris setosa. Below that, we verify that it has the expected number of true and false values.

In [ ]:
# Create a vector of iris species names.
labels = iris.target.apply(lambda i: iris.target_names[i]).rename("label")

# Create a vector of true/false values based on whether the label is "setosa".
y_setosa_mask = (labels == "setosa")
y_setosa_mask.value_counts()

Logistic regression uses the sigmoid function to convert predictions into probabilities between 0 and 1. Visualizing this curve helps you see how the model decides between the two classes.

In [ ]:
t = np.linspace(-5, 5, 100)
sigmoid = 1 / (1 + np.exp(-t))
plt.plot(t, sigmoid, linewidth=2)
plt.xlabel("Input")
plt.ylabel("Probability")
plt.grid()

In the example plot above, where the curve is centered at zero on the x-axis, an input of 2 means the classifier would associate an approximately 85% probability of that sample being in the "positive" class. (The curve used by the classifier will not necessarily be centered at zero.)

In [ ]:
log_reg = linear_model.LogisticRegression()
log_reg.fit(X, y_setosa_mask)

Let's plot the linear decision boundary for this logistic regression model, looking just at petal sizes. (Note that the model was trained on all 4 features.) The decision boundary is based off the model's learned weights and is what determines whether a sample is classified in the positive or negative class. These weights are accessed with the `coef_` and `intercept_` instance variables.

You are not expected to understand how these weights are being used to draw this line; the point of this plot is to demonstrate that the model maintains these learned weights and they express a linear relationship with the data.

In [ ]:
_, axes = plt.subplots(nrows=1, ncols=1)
X.plot.scatter(
    "petal_length",
    "petal_width",
    c=y_setosa_mask,
    colormap="viridis",
    colorbar=False,
    ax=axes,
    title="Petal Size Comparison"
)

decision_boundary = -(
    (log_reg.coef_[0, 2] * np.array(axes.get_xlim()) + log_reg.intercept_[0])
    / log_reg.coef_[0, 3]
)
axes.plot(axes.get_xlim(), decision_boundary)
axes.set_ylim(-1, 3)

Let's make predictions; again, we are using the entirety of a known-clean dataset for both training and testing, so we expect the results to be quite good.

In [ ]:
y_pred_log = log_reg.predict(X)

The confusion matrix illustrates the cleaniless of the data.

In [ ]:
metrics.ConfusionMatrixDisplay.from_predictions(
    y_setosa_mask,
    y_pred_log,
    normalize="true",
    values_format=".0%",
    cmap="Blues",
    colorbar=False
)

#### Self-Check
What does each cell in a confusion matrix represent?

### Multiclass Logistic Regression Using Softmax

When there are more than two classes, logistic regression uses the softmax function to assign probabilities to each possible class. This allows the model to handle problems like predicting which species a flower belongs to among three options.

We will now instantiate another logistic regression classifier. This time we will do multiclass classification (since there are 3 Iris species). In order to do so, the model will use softmax regression.

In [ ]:
log_reg = linear_model.LogisticRegression()
log_reg.fit(X, y)
y_pred_log = log_reg.predict(X)

What happened here? Assuming you can see the above warning, this output means that this model was not able to settle on a set of weights, even after training for the maximum number of iterations. Because logistic regression applies a nonlinear transformation (the sigmoid or softmax function) to its features, it is often more sensitive to feature scaling (or lack thereof) than linear regression. The message suggests scaling our features first.



Let's use the `StandardScaler` transformer to scale the features. (We could have scaled the data at the very beginning, but seeing these warnings and learning how to respond to them can be instructive.)

In [ ]:
X_scaled = preprocessing.StandardScaler().fit_transform(X)

In [ ]:
log_reg = linear_model.LogisticRegression()
log_reg.fit(X_scaled, y)
y_pred_log = log_reg.predict(X_scaled)

Now the model fit the data without issue. Let's display another confusion matrix to see the results.

For a multiclass problem such as this one, the confusion matrix shows how often the model predicts each Iris species correctly or incorrectly. Interpreting this matrix helps you identify which classes are easiest or hardest for the model to distinguish.

In [ ]:
metrics.ConfusionMatrixDisplay.from_predictions(
    y,
    y_pred_log,
    normalize="true",
    values_format=".0%",
    cmap="Blues",
    colorbar=False
)

As expected, the model did perfectly on Iris setosa (label `0`), but got a little mixed up with the other two species, which we saw at the beginning had a bit of overlap between them in the scatter plot.

## What's Next?

You are encouraged to try other linear regression models in scikit-learn:

- `linear_model.Lasso` for L1 regularization
- `linear_model.Ridge` for L2 regularization
- `linear_model.ElasticNet` for combined L1 and L2

See this week's Canvas module for more information about regularization and why it is important.